# Day 051 — Exercise 5: The ChatApp Class

**What you'll build:** `ChatApp` — the logic core stored in `st.session_state` and reused across reruns. Methods: `__init__(model, system_prompt)`, `send(user_text) -> str`, `stats() -> dict`, `transcript() -> str`, `reset()`.

**Why it matters:** This is the whole pattern in one object. The Streamlit file stays a thin shell — it instantiates one `ChatApp`, calls `send()` on submit, and renders `stats()`/`transcript()`. Every function you wrote in Exercises 1–4 is wired together here, and *none* of it depends on Streamlit, so it's fully testable in this notebook.

## Provided: All Helper Functions (Exercises 1–4)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import ollama


def init_session(state: dict) -> dict:
    """
    Idempotently initialise a Streamlit-style session_state dict.

    Streamlit reruns the WHOLE script top-to-bottom on every interaction, so
    initialisation must never overwrite existing data. Only set a key if absent.

    Ensures keys:
        'messages'  -> list of {'role', 'content'} dicts (starts empty)
        'settings'  -> {'model', 'temperature', 'system_prompt'}
    Returns the same dict, mutated in place.
    """
    if 'messages' not in state:
        state['messages'] = []
    if 'settings' not in state:
        state['settings'] = {
            'model': 'llama3.2',
            'temperature': 0.7,
            'system_prompt': 'You are a helpful assistant.',
        }
    return state


def add_message(state: dict, role: str, content: str) -> dict:
    """Append a {'role', 'content'} message to state['messages']; return it."""
    if role not in ('user', 'assistant', 'system'):
        raise ValueError(f'invalid role: {role!r}')
    msg = {'role': role, 'content': content}
    state['messages'].append(msg)
    return msg


def reset_messages(state: dict) -> None:
    """Clear the conversation but keep settings (a 'Clear chat' button)."""
    state['messages'] = []


def validate_user_input(text: str, max_chars: int = 2000) -> tuple[bool, str]:
    """
    Validate raw text from an st.chat_input / st.text_area widget before it is
    sent to the model.

    Returns (is_valid, result):
      - empty/whitespace : (False, 'Please enter a message.')
      - too long         : (False, 'Message too long (max N chars).')
      - valid            : (True, cleaned_text)   # stripped
    """
    cleaned = text.strip()
    if not cleaned:
        return (False, 'Please enter a message.')
    if len(cleaned) > max_chars:
        return (False, f'Message too long (max {max_chars} chars).')
    return (True, cleaned)


def clamp(value: float, lo: float, hi: float) -> float:
    """Clamp a widget value into [lo, hi]. st.slider bounds live input, but a
    value restored from session_state or a URL param may be out of range."""
    return max(lo, min(hi, value))


def build_settings(model: str, temperature: float, system_prompt: str) -> dict:
    """
    Assemble a validated settings dict from sidebar widget values.
    - temperature clamped to [0.0, 1.0]
    - system_prompt stripped; empty falls back to a default
    """
    sp = system_prompt.strip() or 'You are a helpful assistant.'
    return {
        'model': model,
        'temperature': float(clamp(temperature, 0.0, 1.0)),
        'system_prompt': sp,
    }


def build_messages(state: dict, user_text: str) -> list:
    """
    Build the messages list for ollama.chat:
        [system_prompt] + prior conversation + new user turn.
    Reads the system prompt from state['settings']. Does NOT mutate state.
    """
    settings = state.get('settings', {})
    system_prompt = settings.get('system_prompt', 'You are a helpful assistant.')
    messages = [{'role': 'system', 'content': system_prompt}]
    messages.extend(state.get('messages', []))
    messages.append({'role': 'user', 'content': user_text})
    return messages


def chat_with_history(state: dict, user_text: str, model: str = 'llama3.2') -> str:
    """
    Send the full conversation to Ollama and return the assistant's reply.
    Reads temperature from state['settings']. Returns a fallback string if
    Ollama is unavailable so the app never crashes on a model error.
    """
    settings = state.get('settings', {})
    temperature = settings.get('temperature', 0.7)
    messages = build_messages(state, user_text)
    try:
        response = ollama.chat(
            model=model,
            messages=messages,
            options={'temperature': temperature},
        )
        return response['message']['content'].strip()
    except Exception as e:
        return f'[Model unavailable: {e}]' 


def format_transcript(messages: list) -> str:
    """
    Render the conversation as a plain-text transcript for st.download_button.
    One block per turn as 'ROLE: content'. System messages are skipped.
    """
    lines = []
    for m in messages:
        role = m.get('role', '')
        if role == 'system':
            continue
        lines.append(f"{role.upper()}: {m.get('content', '')}")
    return '\n\n'.join(lines)


def chat_stats(messages: list) -> dict:
    """
    Compute display metrics for st.metric widgets. System messages excluded.
    Returns: {'total', 'user', 'assistant', 'chars'}.
    """
    non_system = [m for m in messages if m.get('role') != 'system']
    user = sum(1 for m in non_system if m.get('role') == 'user')
    assistant = sum(1 for m in non_system if m.get('role') == 'assistant')
    chars = sum(len(m.get('content', '')) for m in non_system)
    return {
        'total': len(non_system),
        'user': user,
        'assistant': assistant,
        'chars': chars,
    }

## Your Implementation

In [ ]:
class ChatApp:
    """
    Logic core of the Streamlit chat app. One instance per session,
    reused across reruns. The UI only calls these methods.
    """

    def __init__(self, model: str = 'llama3.2',
                 system_prompt: str = 'You are a helpful assistant.'):
        # TODO: self.state = {}; init_session(self.state)
        # TODO: self.state['settings']['model'] = model
        # TODO: self.state['settings']['system_prompt'] = system_prompt
        # TODO: self.model = model
        pass

    def send(self, user_text: str) -> str:
        """Validate -> append user -> call model -> append assistant -> return reply.
        On invalid input, return the error string and append NOTHING."""
        # TODO: ok, result = validate_user_input(user_text)
        # TODO: if not ok: return result
        # TODO: add_message(self.state, 'user', result)
        # TODO: reply = chat_with_history(self.state, result, self.model)
        # TODO: add_message(self.state, 'assistant', reply)
        # TODO: return reply
        pass

    def stats(self) -> dict:
        # TODO: return chat_stats(self.state['messages'])
        pass

    def transcript(self) -> str:
        # TODO: return format_transcript(self.state['messages'])
        pass

    def reset(self) -> None:
        # TODO: reset_messages(self.state)
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: ChatApp has the four public methods
    try:
        assert 'ChatApp' in globals()
        for m in ('send', 'stats', 'transcript', 'reset'):
            assert hasattr(ChatApp, m), f'missing method: {m}'
        passed += 1; print('✅ Check 1: ChatApp has send/stats/transcript/reset')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: send() appends one user + one assistant message
    try:
        app = ChatApp()
        reply = app.send('Reply with the single word: pong')
        assert isinstance(reply, str) and len(reply) > 0, 'send must return a non-empty reply'
        s = app.stats()
        assert s['total'] == 2, f"expected 2 messages after one send, got {s['total']}"
        assert s['user'] == 1 and s['assistant'] == 1, 'expected 1 user + 1 assistant'
        passed += 1; print(f'✅ Check 2: send() appended user + assistant ({len(reply)} char reply)')
    except Exception as e:
        print(f'❌ Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: invalid input returns an error and appends nothing
    try:
        app = ChatApp()
        out = app.send('   ')
        assert isinstance(out, str) and 'enter a message' in out.lower(), f'expected error msg, got {out!r}'
        assert app.stats()['total'] == 0, 'invalid input must not append messages'
        passed += 1; print('✅ Check 3: invalid input -> error, nothing appended')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: transcript() reflects the conversation
    try:
        app = ChatApp()
        app.state['messages'] = [
            {'role': 'user', 'content': 'hello'},
            {'role': 'assistant', 'content': 'hi back'},
        ]
        t = app.transcript()
        assert 'USER: hello' in t, f'transcript missing user turn:\n{t}'
        passed += 1; print('✅ Check 4: transcript() renders the conversation')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: reset() clears the conversation
    try:
        app = ChatApp()
        app.state['messages'] = [{'role': 'user', 'content': 'x'}]
        app.reset()
        assert app.stats()['total'] == 0, 'reset() must clear messages'
        assert 'settings' in app.state, 'reset() must keep settings'
        passed += 1; print('✅ Check 5: reset() clears the conversation')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class ChatApp:
    """
    The logic core of the Streamlit chat app. One instance is stored in
    st.session_state and reused across reruns. The Streamlit layer only calls
    these methods and renders their return values — no business logic in the UI.

    Usage (inside app.py):
        if 'app' not in st.session_state:
            st.session_state.app = ChatApp()
        app = st.session_state.app
        reply = app.send(prompt)          # on chat_input submit
        st.metric('Messages', app.stats()['total'])
    """

    def __init__(self, model: str = 'llama3.2',
                 system_prompt: str = 'You are a helpful assistant.'):
        self.state = {}
        init_session(self.state)
        self.state['settings']['model'] = model
        self.state['settings']['system_prompt'] = system_prompt
        self.model = model

    def send(self, user_text: str) -> str:
        """
        Validate -> append user turn -> call model -> append assistant turn.
        Returns the assistant reply, or a validation error string (in which
        case NOTHING is appended to the conversation).
        """
        ok, result = validate_user_input(user_text)
        if not ok:
            return result
        add_message(self.state, 'user', result)
        reply = chat_with_history(self.state, result, self.model)
        add_message(self.state, 'assistant', reply)
        return reply

    def stats(self) -> dict:
        return chat_stats(self.state['messages'])

    def transcript(self) -> str:
        return format_transcript(self.state['messages'])

    def reset(self) -> None:
        reset_messages(self.state)
```

**Why this works:** `ChatApp` owns its own `state` dict — the same shape `st.session_state` would hold — so the Streamlit layer just stores one instance in `st.session_state.app` and calls methods on it. `send()` composes the whole turn (validate → user → model → assistant) and only commits messages when the input is valid, so a stray Enter key never pollutes the history. Because nothing here imports Streamlit, the entire app is testable in a plain notebook — which is exactly what these five exercises did.
</details>